# Notebook 02 — LoRA: fine-tune <1% of the weights

**Workshop:** Agentic AI for Scientists — Week 4 (Post-Training & Deployment)
**Block:** parameter-efficient fine-tuning (20 min)
**Goal:** Reach (almost) the same behavior as full SFT while **freezing the base model** and training only small **low-rank adapter** matrices injected next to the attention/MLP projections.

**The idea in one line:** instead of changing a big weight matrix `W`, we *add* a tiny correction `W + B·A`, where `A` and `B` are skinny (rank `r`) — so we train `A` and `B` (a few hundred K params) and leave `W` frozen.

```
        full FT:   update W            (millions of params, big gradients + optim states)
        LoRA   :   W  +  B @ A         (train only A,B  — rank r, e.g. 32)
                   frozen   trainable
```

**Why it matters:** gradients and optimizer states (VRAM buckets 2 & 3 from NB01) now exist only for the adapters. Same idea as full SFT — a fraction of the memory, and the saved adapter is ~5-20 MB instead of ~1.2 GB.

## 0. Setup & GPU check

In [ ]:
%pip install -q unsloth
%pip install -q --no-deps "trl<0.24" peft accelerate bitsandbytes datasets

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run."
)
gpu = torch.cuda.get_device_properties(0)
print(f"GPU            : {gpu.name}")
print(f"Total VRAM     : {gpu.total_memory / 1024**3:.1f} GB")
print(f"Compute Cap.   : {gpu.major}.{gpu.minor}  "
      f"({'Ampere+ (Flash-Attention 2 path)' if gpu.major >= 8 else 'pre-Ampere (T4): works, slower attention path'})")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()}")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

---
## 1. Load base model, then attach LoRA adapters

For LoRA the base model can stay in full precision (we set `load_in_4bit=False`; flip it to `True` and you have **QLoRA** — that's literally NB03). `get_peft_model` injects the trainable adapters into the named projection layers.

- `r=32` — adapter rank (capacity). Higher = more expressive, more params.
- `lora_alpha=16` — scaling. The effective update is scaled by `alpha/r`.
- `target_modules` — which weight matrices get an adapter (all attention + MLP projections).

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-0.6B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=False,         # full-precision base + LoRA. (True -> QLoRA, see NB03)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",   # recompute activations -> shrinks bucket 4
    random_state=3407,
)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {n_total/1e6:.1f}M")
print(f"Trainable params: {n_train/1e6:.3f}M  ({100*n_train/n_total:.2f}%)  <-- LoRA adapters only")

---
## 2. Dataset (identical to NB01)

Same prepared JSONL, same chat template. The *only* thing changing between these notebooks is the training method — that's deliberate, so any quality difference is attributable to the method, not the data.

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset, Dataset

SYSTEM_PROMPT = (
    "You are a clinical assistant. Read the medical question and respond with a JSON object "
    "containing: disease, patient_info, symptoms (list), treatment (list), answer_summary."
)

def load_prepared(split):
    p = Path(f"data/medquad_{split}.jsonl")
    if p.exists():
        return Dataset.from_list([json.loads(l) for l in p.read_text().splitlines() if l.strip()])
    return None

train_ds, eval_ds = load_prepared("train"), load_prepared("validation")
if train_ds is None:
    raw = load_dataset("lavita/MedQuAD", split="train").filter(lambda r: r.get("answer"))
    raw = raw.shuffle(seed=3407).select(range(200))
    rows = [{"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": r["question"]},
        {"role": "assistant", "content": r["answer"][:800]}]} for r in raw]
    train_ds, eval_ds = Dataset.from_list(rows[:160]), Dataset.from_list(rows[160:])

def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}
train_ds, eval_ds = train_ds.map(to_text), eval_ds.map(to_text)
print(f"train: {len(train_ds)} | eval: {len(eval_ds)}")

---
## 3. Train

Note the **higher learning rate (2e-4, 10x NB01)**. With the base frozen there's no catastrophic-forgetting risk, and the small adapters need a stronger signal to move. Everything else matches NB01 so the comparison is fair.

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

cfg = SFTConfig(
    output_dir="outputs_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,                  # DEMO budget (match NB01 for a fair comparison)
    learning_rate=2e-4,            # 10x NB01 — adapters can take it
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="no",
    seed=3407,
    report_to="none",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                     train_dataset=train_ds, eval_dataset=eval_ds, args=cfg)

torch.cuda.reset_peak_memory_stats()
stats = trainer.train()
peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nFinal train loss : {stats.training_loss:.4f}")
print(f"Peak VRAM        : {peak:.2f} GB  (compare to NB01's full-FT peak)")

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
Path("data").mkdir(exist_ok=True)
Path("data/run_lora.json").write_text(json.dumps(
    {"method": "lora", "trainable_pct": round(100*n_train/n_total, 2), "peak_vram_gb": round(peak, 2),
     "final_loss": round(float(stats.training_loss), 4), "steps": stats.global_step}, indent=2))

---
## 4. Eyeball check + save **just the adapter**

The saved artifact is the adapter only — a few MB. To deploy you load the base model and attach this adapter (or *merge* it, which NB03 demonstrates).

In [ ]:
from transformers import TextStreamer
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What causes type 2 diabetes and how is it managed?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
_ = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.3, streamer=TextStreamer(tokenizer, skip_prompt=True))

In [ ]:
model.save_pretrained("qwen3-medquad-lora")       # adapter only (~MB)
tokenizer.save_pretrained("qwen3-medquad-lora")
import os
print("Adapter saved. Size on disk:")
for f in os.listdir("qwen3-medquad-lora"):
    sz = os.path.getsize(os.path.join("qwen3-medquad-lora", f)) / 1024**2
    print(f"  {f:32s} {sz:7.2f} MB")

## What you learned

Same behavior, ~1% of the trainable parameters, a fraction of the VRAM, and a deployable artifact measured in megabytes. LoRA is the default for most real fine-tuning.

**Next — `03_qlora.ipynb`:** push memory even lower by **quantizing the frozen base model to 4-bit**. This is what lets people fine-tune 7B-70B models on a single consumer GPU.